In [ ]:
!pip install requests pandas tqdm -q
import requests, pandas as pd, time, re, os, datetime
from tqdm.notebook import tqdm
from google.colab import drive
print('✅ Ready')

✅ Ready


In [ ]:
drive.mount('/content/drive')
OUTPUT_DIR = '/content/drive/MyDrive/PHI_Research/'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'✅ Drive mounted → {OUTPUT_DIR}')

Mounted at /content/drive
✅ Drive mounted → /content/drive/MyDrive/PHI_Research/


In [ ]:
# ── Configuration ─────────────────────────────────────────────

SUBREDDITS = {
    'experts':  ['blind', 'accessibility', 'disability'],
    'patients': ['diabetes_t1', 'dexcom', 'type1diabetes'],
    'tech':     ['AppleWatch', 'Garmin', 'fitbit', 'ouraring'],
}
ALL_SUBREDDITS = [s for group in SUBREDDITS.values() for s in group]

TECH_TERMS = [
    'update', 'graph', 'trend', 'pump', 'bolus',
    'sync', 'dashboard', 'scale', 'pairing'
]

BARRIER_TERMS = [
    'VoiceOver', 'TalkBack', 'unlabeled', 'tiny', 'contrast',
    'inaccessible', "can't read", 'haptic', 'buttons', 'touchscreen'
]

# Broad accessibility signals — post must match at least one
ACCESSIBILITY_SIGNALS = [
    'voiceover', 'talkback', 'screen reader', 'screenreader',
    'nvda', 'jaws', 'narrator', 'inaccessible', 'accessibility',
    'accessible', 'unlabeled', 'unreadable', "can't read",
    'cannot read', 'hard to read', 'blind', 'low vision',
    'visually impaired', 'haptic', 'tremor', 'dexterity',
    'one hand', 'arthritis', 'deaf', 'hearing', 'caption',
    'workaround', 'after update', 'broke', 'regression',
    'used to work', 'stopped working', 'privacy', 'sighted',
    'contrast', 'font size', 'text size', 'zoom',
]

# Hard exclusions
NOISE = [
    r'vo2\s*max.*tiny', r'stainless steel', r'solo loop',
    r'for sale|\bwts\b|\bwtb\b|\bfs\b', r'battery life',
    r'which (color|band|strap|size)', r'how does .{0,20} look',
    r'discount|promo|coupon',
]

START_DATE   = '2018-01-01'
MIN_COMMENTS = 3
LIMIT        = 100
DELAY        = 0.3   # ⚡ much faster

SUB_TO_GROUP = {
    **{s: 'experts'  for s in SUBREDDITS['experts']},
    **{s: 'patients' for s in SUBREDDITS['patients']},
    **{s: 'tech'     for s in SUBREDDITS['tech']},
}

# 90 total API calls = ~0.3s each = ~30 seconds for scraping!
total_calls = len(ALL_SUBREDDITS) * len(BARRIER_TERMS)
print(f'Total API calls  : {total_calls}')
print(f'Est. scrape time : ~{round(total_calls * DELAY / 60, 1)} min')
print(f'(Comments fetched separately after)')

Total API calls  : 100
Est. scrape time : ~0.5 min
(Comments fetched separately after)


In [ ]:
# ── API + Filter Functions ─────────────────────────────────────

POSTS_URL    = 'https://arctic-shift.photon-reddit.com/api/posts/search'
COMMENTS_URL = 'https://arctic-shift.photon-reddit.com/api/comments/search'

def fetch_posts(subreddit, term, after=START_DATE, limit=LIMIT):
    """Search barrier term in post titles only."""
    try:
        r = requests.get(POSTS_URL, params={
            'subreddit': subreddit,
            'title':     term,
            'after':     after,
            'limit':     limit,
        }, timeout=20)
        if r.status_code == 200:
            return r.json().get('data') or []
    except Exception as e:
        print(f'  ⚠️ [{subreddit}|{term}]: {e}')
    return []


def fetch_comments_batch(post_ids, limit=5):
    """Fetch comments for a list of post IDs. Returns dict {pid: comments_str}"""
    results = {}
    for pid in tqdm(post_ids, desc='Fetching comments', leave=False):
        try:
            r = requests.get(COMMENTS_URL, params={
                'link_id': f't3_{pid}',
                'limit':   limit,
            }, timeout=15)
            if r.status_code == 200:
                comments = r.json().get('data') or []
                texts = [
                    c.get('body', '') for c in comments
                    if c.get('body') not in ['', '[deleted]', '[removed]', None]
                ]
                results[pid] = ' ||| '.join(texts[:limit])
            else:
                results[pid] = ''
        except:
            results[pid] = ''
        time.sleep(0.2)
    return results


def parse_date(val):
    try:
        return datetime.datetime.utcfromtimestamp(int(val)).strftime('%Y-%m-%d')
    except:
        return str(val)


def is_accessible(text):
    t = text.lower()
    return any(s in t for s in ACCESSIBILITY_SIGNALS)


def is_noise(text):
    t = text.lower()
    return any(re.search(p, t) for p in NOISE)


def infer_brand(text):
    t = text.lower()
    if any(k in t for k in ['apple watch', 'voiceover', 'watchos', 'iphone', 'ios']):
        return 'Apple'
    elif any(k in t for k in ['dexcom', 'g6', 'g7', 'clarity', 'cgm']):
        return 'Dexcom'
    elif any(k in t for k in ['garmin', 'fenix', 'forerunner']):
        return 'Garmin'
    elif any(k in t for k in ['fitbit', 'versa', 'sense', 'charge']):
        return 'Fitbit'
    elif any(k in t for k in ['oura', 'oura ring']):
        return 'Oura'
    return 'Other'


# Sanity check
print('Sanity check...')
test = fetch_posts('blind', 'VoiceOver')
print(f'r/blind + VoiceOver → {len(test)} posts')
if test:
    print(f'Sample: {test[0].get("title")}')
    print('✅ API working!')
else:
    print('⚠️ 0 posts — check connection')

Sanity check...
r/blind + VoiceOver → 100 posts
Sample: VoiceOver for Mac - 3rd Party Tutorial Apps?
✅ API working!


In [ ]:
# ── PHASE 1: Fast Scrape (no comments yet) ────────────────────
# Searches each BARRIER TERM across all subreddits
# ~90 API calls, ~30 seconds total

raw_posts  = {}   # pid → post dict
pid_meta   = {}   # pid → {search_term, subreddit}

jobs = [
    (sub, barrier)
    for sub     in ALL_SUBREDDITS
    for barrier in BARRIER_TERMS
]

print(f'⚡ Phase 1: Scraping {len(jobs)} queries...\n')

for sub, barrier in tqdm(jobs, desc='Scraping'):
    posts = fetch_posts(sub, barrier)
    time.sleep(DELAY)

    for post in posts:
        pid = post.get('id', '')
        if not pid or pid in raw_posts:
            continue
        raw_posts[pid] = post
        pid_meta[pid]  = {'search_term': barrier, 'source_subreddit': sub}

print(f'\n✅ Phase 1 done!')
print(f'   Raw posts fetched (before filters): {len(raw_posts)}')

⚡ Phase 1: Scraping 100 queries...



Scraping:   0%|          | 0/100 [00:00<?, ?it/s]


✅ Phase 1 done!
   Raw posts fetched (before filters): 1174


In [ ]:
# ── PHASE 2: Filter ───────────────────────────────────────────
# Apply all filters to the raw posts

filtered = []
stats = {'comments': 0, 'noise': 0, 'not_accessible': 0, 'kept': 0}

for pid, post in raw_posts.items():
    num_comments = int(post.get('num_comments', 0))
    if num_comments <= MIN_COMMENTS:
        stats['comments'] += 1
        continue

    title    = post.get('title',    '') or ''
    selftext = post.get('selftext', '') or ''
    full_text = (title + ' ' + selftext).strip()

    if is_noise(full_text):
        stats['noise'] += 1
        continue

    if not is_accessible(full_text):
        stats['not_accessible'] += 1
        continue

    stats['kept'] += 1
    filtered.append((pid, post, full_text, title, selftext))

print(f'Filter results:')
print(f'  Too few comments : {stats["comments"]}')
print(f'  Noise/off-topic  : {stats["noise"]}')
print(f'  Not accessible   : {stats["not_accessible"]}')
print(f'  ✅ Kept           : {stats["kept"]}')

Filter results:
  Too few comments : 612
  Noise/off-topic  : 18
  Not accessible   : 171
  ✅ Kept           : 373


In [ ]:
# ── PHASE 3: Fetch Comments for Kept Posts Only ───────────────
# Much faster — only fetching comments for posts we actually keep

kept_pids = [pid for pid, *_ in filtered]
print(f'Fetching comments for {len(kept_pids)} posts...')
comments_map = fetch_comments_batch(kept_pids, limit=5)
print(f'✅ Comments fetched!')

In [ ]:
# ── PHASE 4: Build Final DataFrame ───────────────────────────

records = []
for pid, post, full_text, title, selftext in filtered:
    meta = pid_meta.get(pid, {})
    sub  = meta.get('source_subreddit', '')

    matched_barriers = [b for b in BARRIER_TERMS if b.lower() in full_text.lower()]
    matched_tech     = [t for t in TECH_TERMS    if t.lower() in full_text.lower()]

    records.append({
        'post_id'          : pid,
        'source_subreddit' : sub,
        'subreddit_group'  : SUB_TO_GROUP.get(sub, 'unknown'),
        'search_term'      : meta.get('search_term', ''),
        'matched_barriers' : ', '.join(matched_barriers),
        'matched_tech'     : ', '.join(matched_tech),
        'title'            : title,
        'selftext'         : selftext,
        'full_text'        : full_text,
        'top_comments'     : comments_map.get(pid, ''),
        'score'            : post.get('score', 0),
        'num_comments'     : int(post.get('num_comments', 0)),
        'upvote_ratio'     : post.get('upvote_ratio', None),
        'created_utc'      : parse_date(post.get('created_utc', '')),
        'author'           : post.get('author', ''),
        'url'              : 'https://reddit.com' + post.get('permalink', ''),
        'flair'            : post.get('link_flair_text', ''),
        'device_brand'     : infer_brand(full_text),
    })

df = pd.DataFrame(records)

print('=' * 50)
print('FINAL DATASET SUMMARY')
print('=' * 50)
print(f'Total posts   : {len(df)}')
print(f'Unique IDs    : {df["post_id"].nunique()}')
print(f'Date range    : {df["created_utc"].min()} → {df["created_utc"].max()}')
print()
print('Per subreddit:')
print(df['source_subreddit'].value_counts().to_string())
print()
print('Per group:')
print(df['subreddit_group'].value_counts().to_string())
print()
print('Device brand:')
print(df['device_brand'].value_counts().to_string())

FINAL DATASET SUMMARY
Total posts   : 373
Unique IDs    : 373
Date range    : 2018-02-05 → 2026-05-23

Per subreddit:
source_subreddit
blind            172
accessibility    100
disability        50
AppleWatch        48
dexcom             1
type1diabetes      1
Garmin             1

Per group:
subreddit_group
experts     322
tech         49
patients      2

Device brand:
device_brand
Other     227
Apple     137
Fitbit      7
Dexcom      2


/tmp/ipykernel_26989/1028537997.py:48: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  return datetime.datetime.utcfromtimestamp(int(val)).strftime('%Y-%m-%d')


In [ ]:
# ── PHASE 5: Feature Engineering for Topic and RQ Labels ─────────────────
# Generate clean_text, preliminary topic_name, and preliminary RQ_Category
# These are rule-based intermediate labels for Stage 2 and can be refined later

import re

def build_analysis_text(row):
    """
    Combine all useful textual fields into one analysis string.
    This gives the classifier more context than title/selftext alone.
    """
    parts = [
        str(row.get("title", "")),
        str(row.get("selftext", "")),
        str(row.get("full_text", "")),
        str(row.get("top_comments", ""))
    ]
    text = " ".join(parts).lower()
    text = re.sub(r"\s+", " ", text).strip()
    return text


def clean_text_basic(text):
    """
    Basic text cleaning for NLP-style processing.
    Keeps letters, numbers, apostrophes, and spaces.
    """
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", "", text)      # remove URLs
    text = re.sub(r"[^a-z0-9\s']", " ", text)       # remove punctuation/special chars
    text = re.sub(r"\s+", " ", text).strip()        # normalize whitespace
    return text


def assign_rq_category(text):
    """
    Assign preliminary research-question category using rule-based heuristics.
    Priority matters here: more specific categories are checked first.
    """
    text = text.lower()

    # 1. Update Trauma
    if any(k in text for k in [
        "update broke", "after update", "since the update", "new update",
        "latest update", "app update", "software update", "version",
        "worked yesterday", "used to work", "stopped working",
        "broke voiceover", "broken after update"
    ]):
        return "Update Trauma"

    # 2. Privacy / Dependency
    if any(k in text for k in [
        "ask someone", "ask my wife", "ask my husband", "ask a friend",
        "need help reading", "someone else reads", "sighted person",
        "audio announces", "announces out loud", "privacy",
        "private information", "embarrassing in public", "read it for me"
    ]):
        return "Privacy/Dependency"

    # 3. Physical / Dexterity Barrier
    if any(k in text for k in [
        "buttons too small", "small buttons", "hard to press", "touchscreen",
        "tap gesture", "double tap", "cannot put on", "can't put on",
        "strap", "band", "dexterity", "tremor", "fine motor",
        "physical difficulty", "hard to wear", "hard to attach"
    ]):
        return "Physical/Dexterity Barrier"

    # 4. Workaround / Hack
    if any(k in text for k in [
        "workaround", "hack", "i use this app instead", "alternative app",
        "i found a fix", "temporary fix", "solution", "3d printed",
        "custom setup", "my workaround", "what worked for me", "i switched to"
    ]):
        return "Workaround/Hack"

    # 5. Data Denial (Visual)
    if any(k in text for k in [
        "voiceover", "talkback", "unlabeled", "can't read", "cannot read",
        "graph", "trend", "chart", "dashboard", "contrast", "tiny text",
        "data visualization", "screen reader", "not accessible",
        "can't see", "cannot see"
    ]):
        return "Data Denial (Visual)"

    return "TBD"


def assign_topic_name(text):
    """
    Assign a preliminary topic label using rule-based themes.
    These are interim themes and can later be replaced by BERTopic output.
    """
    text = text.lower()

    if any(k in text for k in [
        "update broke", "after update", "latest update", "version",
        "stopped working", "used to work"
    ]):
        return "App Updates Breaking Accessibility"

    if any(k in text for k in [
        "buttons too small", "small buttons", "touchscreen", "strap",
        "band", "hard to press", "dexterity", "tremor", "hard to wear",
        "fine motor"
    ]):
        return "Physical Dexterity/Buttons"

    if any(k in text for k in [
        "graph", "trend", "chart", "dashboard", "can't read", "cannot read",
        "voiceover", "talkback", "unlabeled", "contrast", "tiny text",
        "screen reader"
    ]):
        return "Data Visualization Issues"

    if any(k in text for k in [
        "sync", "pairing", "paired", "third party", "apple health",
        "health app", "integration", "connect", "bluetooth"
    ]):
        return "Third-Party App Integration"

    if any(k in text for k in [
        "privacy", "ask someone", "need help reading", "audio announces",
        "read it for me", "sighted person"
    ]):
        return "Privacy and Dependency"

    if any(k in text for k in [
        "workaround", "hack", "fix", "alternative app", "custom setup",
        "3d printed", "what worked for me", "i switched to"
    ]):
        return "Workarounds and Hacks"

    return "TBD"


# Build combined text for analysis
df["analysis_text"] = df.apply(build_analysis_text, axis=1)

# Create cleaned text version for reporting / downstream NLP
df["clean_text"] = df["analysis_text"].apply(clean_text_basic)

# Assign preliminary labels
df["RQ_Category"] = df["clean_text"].apply(assign_rq_category)
df["topic_name"] = df["clean_text"].apply(assign_topic_name)

# Optional: reorder columns so the new fields appear in a cleaner place
preferred_order = [
    "post_id",
    "source_subreddit",
    "subreddit_group",
    "search_term",
    "device_brand",
    "matched_barriers",
    "matched_tech",
    "title",
    "selftext",
    "full_text",
    "top_comments",
    "analysis_text",
    "clean_text",
    "topic_name",
    "RQ_Category",
    "score",
    "num_comments",
    "upvote_ratio",
    "created_utc",
    "author",
    "url",
    "flair"
]

# Keep only columns that exist, in that order
df = df[[col for col in preferred_order if col in df.columns]]

# Quick validation checks
print("\n" + "=" * 50)
print("PHASE 5 LABEL CHECK")
print("=" * 50)

print("\nRQ_Category distribution:")
print(df["RQ_Category"].value_counts(dropna=False).to_string())

print("\nTopic distribution:")
print(df["topic_name"].value_counts(dropna=False).to_string())

print("\nTBD counts:")
print("RQ_Category TBD :", (df["RQ_Category"] == "TBD").sum())
print("topic_name TBD  :", (df["topic_name"] == "TBD").sum())

print("\nSample rows:")
print(df[["source_subreddit", "device_brand", "topic_name", "RQ_Category", "title"]].head(10).to_string(index=False))


PHASE 5 LABEL CHECK

RQ_Category distribution:
RQ_Category
Data Denial (Visual)          147
TBD                            70
Update Trauma                  68
Physical/Dexterity Barrier     41
Workaround/Hack                39
Privacy/Dependency              8

Topic distribution:
topic_name
Data Visualization Issues             180
App Updates Breaking Accessibility     67
TBD                                    62
Physical Dexterity/Buttons             31
Third-Party App Integration            26
Workarounds and Hacks                   7

TBD counts:
RQ_Category TBD : 70
topic_name TBD  : 62

Sample rows:
source_subreddit device_brand                 topic_name                RQ_Category                                                                                                                                                     title
           blind        Apple  Data Visualization Issues       Data Denial (Visual)                                                              

In [ ]:
# ── Preview 5 posts ───────────────────────────────────────────
print('SAMPLE POSTS')
print('=' * 65)
sample = df.sample(min(5, len(df)), random_state=42)
for _, row in sample.iterrows():
    print(f"\n[r/{row['source_subreddit']}] | {row['created_utc']} | {row['num_comments']} comments | {row['device_brand']}")
    print(f"BARRIERS : {row['matched_barriers']}")
    print(f"TECH     : {row['matched_tech']}")
    print(f"TITLE    : {str(row['title'])[:120]}")
    print(f"TEXT     : {str(row['selftext'])[:200]}")
    print('-' * 65)

In [ ]:
def calculate_discussion_depth(row):
    """
    Enhanced filter: Scores the quality of the discussion.
    Considers comment count, total text length, and presence of help/workaround keywords.
    """
    comments = str(row['top_comments'])
    if row['num_comments'] <= 3 or not comments:
        return 0.0

    # Factor 1: Engagement (normalized comment count)
    engagement = min(row['num_comments'] / 20, 1.0)

    # Factor 2: Text Richness (average length of discussion)
    richness = min(len(comments) / 1000, 1.0)

    # Factor 3: Insight Signals (looking for solutions/workarounds)
    insight_keywords = ['try', 'fixed', 'setting', 'workaround', 'instead', 'update', 'help']
    insight_score = sum(1 for word in insight_keywords if word in comments.lower()) / len(insight_keywords)

    return round((engagement * 0.3) + (richness * 0.3) + (insight_score * 0.4), 2)

# Apply the enhancement
df['discussion_depth_score'] = df.apply(calculate_discussion_depth, axis=1)

# Filter for high-depth discussions only
df_high_depth = df[df['discussion_depth_score'] > 0.15].sort_values(by='discussion_depth_score', ascending=False)

print(f'Original Records: {len(df)}')
print(f'High-Quality Discussion Records: {len(df_high_depth)}')

# Show the top 5 most 'insightful' discussions
print('\n--- TOP 5 DISCUSSIONS BY DEPTH SCORE ---')
print(df_high_depth[['title', 'num_comments', 'discussion_depth_score', 'source_subreddit']].head(5).to_string(index=False))

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF
from sklearn.feature_extraction import text
import numpy as np

# Expanded stop words to surface specific technical barriers
custom_stops = ['voiceover', 'talkback', 'just', 'like', 'use', 'using', 'used', 'app', 'apps', 'phone', 'reddit', 'post', 'comments', 'screen', 'reader', 'accessibility', 'accessible']
stop_words = text.ENGLISH_STOP_WORDS.union(custom_stops)

documents = df_high_depth['full_text'].fillna('') + ' ' + df_high_depth['top_comments'].fillna('')

# Vectorize with more strict constraints to ignore universal terms
tfidf_vectorizer = TfidfVectorizer(max_df=0.6, min_df=3, stop_words=list(stop_words))
tfidf = tfidf_vectorizer.fit_transform(documents)

# NMF provides better sparsity/separation for this type of qualitative data
nmf = NMF(n_components=5, random_state=42, init='nndsvd')
nmf.fit(tfidf)

def display_topics(model, feature_names, no_top_words):
    topics_list = []
    for topic_idx, topic in enumerate(model.components_):
        top_words = [feature_names[i] for i in topic.argsort()[:-no_top_words - 1:-1]]
        topics_list.append({"Topic": topic_idx + 1, "Top Words": ", ".join(top_words)})
    return pd.DataFrame(topics_list)

tfidf_feature_names = tfidf_vectorizer.get_feature_names_out()
topics_df = display_topics(nmf, tfidf_feature_names, 10)

print("--- REFINED ACCESSIBILITY THEMES (NMF) ---")
display(topics_df)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Assign refined topics
topic_values = nmf.transform(tfidf)
df_high_depth['dominant_topic'] = topic_values.argmax(axis=1) + 1

# Map topic numbers to descriptive labels based on the NMF results
topic_labels = {
    1: "Desktop/Screen Reader",
    2: "Visual/Contrast/WCAG",
    3: "Wearables/Haptics",
    4: "Social/Physical Barriers",
    5: "Mobile/Android/Braille"
}

# Prepare data for plotting
topic_counts = df_high_depth['dominant_topic'].map(topic_labels).value_counts().sort_index()

plt.figure(figsize=(12, 6))
sns.set_style("whitegrid")
ax = sns.barplot(x=topic_counts.index, y=topic_counts.values, palette="viridis", hue=topic_counts.index, legend=False)

plt.title('Distribution of Refined Accessibility Themes', fontsize=15)
plt.xlabel('Accessibility Theme', fontsize=12)
plt.ylabel('Number of High-Depth Posts', fontsize=12)
plt.xticks(rotation=15)

# Add count labels on top of bars
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center', fontsize=11, color='black', xytext=(0, 5),
                textcoords='offset points')

plt.tight_layout()
plt.show()

print("Refined Topic Distribution:")
print(df_high_depth['dominant_topic'].value_counts().sort_index())

# Show sample for a more specific topic (e.g., Topic 2)
if not df_high_depth[df_high_depth['dominant_topic'] == 2].empty:
    print("\n--- Sample from Topic 2 ---")
    print(df_high_depth[df_high_depth['dominant_topic'] == 2]['title'].head(3).tolist())

# 📑 Research Documentation: PHI Accessibility Barrier Mining

## 1. Methodology Overview
This project uses the **Arctic Shift API** (a high-performance Reddit archive) to mine accessibility barriers within Personal Health Informatics (PHI). The workflow is divided into four main phases:

1.  **Fast Scrape**: Targeted keyword searches across 10 specialized subreddits (Experts, Patients, and Tech Users).
2.  **Multistage Filtering**: Posts are filtered by comment volume (>3), noise removal (sales/promos), and accessibility signal verification.
3.  **Discussion Enrichment**: Top-level comments are fetched for high-potential posts to capture community workarounds and technical depth.
4.  **Discussion Depth Scoring**: A custom heuristic evaluates the 'richness' of the conversation based on engagement, text length, and solution-oriented keywords.

## 2. Data Dictionary (`PHI_raw.csv`)

| Column | Meaning |
| :--- | :--- |
| `post_id` | Unique Reddit identifier (e.g., `t3_xxxx`). |
| `source_subreddit` | The subreddit where the post was found (e.g., r/blind, r/AppleWatch). |
| `subreddit_group` | Categorization: `experts`, `patients`, or `tech`. |
| `matched_barriers` | Barrier terms found in the text (e.g., 'VoiceOver', 'Haptic'). |
| `matched_tech` | Technical terms found (e.g., 'Sync', 'Graph'). |
| `top_comments` | Concatenated text of the top 5 most relevant comments (separated by `|||`). |
| `score` | The net upvotes (Upvotes minus Downvotes) of the post. |
| `num_comments` | Total count of all comments and sub-comments in the thread. |
| `upvote_ratio` | Percentage of positive engagement (0.0 to 1.0). |
| `discussion_depth_score` | A 0-1 score representing the qualitative 'richness' of the technical discussion. |
| `device_brand` | Inferred hardware manufacturer based on text analysis (e.g., Apple, Dexcom, Garmin). |
| `dominant_topic` | The cluster ID (1-5) assigned via NMF Topic Modeling. |

## 3. Usage Notes
- **Privacy**: This file contains raw usernames. Ensure de-identification occurs before sharing datasets externally.
- **Date Range**: Covers posts from January 2018 to the present day.

In [ ]:
# Assign refined topics
topic_values = nmf.transform(tfidf)
df_high_depth['dominant_topic'] = topic_values.argmax(axis=1) + 1

print("Refined Topic Distribution:")
print(df_high_depth['dominant_topic'].value_counts().sort_index())

# Show sample for a more specific topic (e.g., Topic 2)
if not df_high_depth[df_high_depth['dominant_topic'] == 2].empty:
    print("\n--- Sample from Topic 2 ---")
    print(df_high_depth[df_high_depth['dominant_topic'] == 2]['title'].head(3).tolist())

In [ ]:
# ── Save ──────────────────────────────────────────────────────
RAW_OUTPUT = os.path.join(OUTPUT_DIR, 'PHI_raw.csv')
df.to_csv(RAW_OUTPUT, index=False, encoding='utf-8-sig')
print(f'✅ Saved {len(df)} posts → {RAW_OUTPUT}')
print('⚠️  Contains raw usernames — do not share publicly')
print('Next → 02_Clean_Redact.ipynb')